# **Instalação do Groq**

In [1]:
%pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.1 MB/s eta 0:00:00


# **Código da API**

In [2]:
import os
os.environ['GROQ_API_KEY'] = "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"

# **Inicialização do Groq**

In [3]:
import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in the field of natural language processing (NLP) and have numerous applications in various industries. The importance of fast language models can be attributed to the following reasons:

1. **Efficient Processing**: Fast language models can process large amounts of text data quickly, making them ideal for applications where speed is essential, such as real-time chatbots, voice assistants, and sentiment analysis.
2. **Improved User Experience**: Fast language models enable faster response times, which leads to a better user experience. For example, in virtual assistants like Siri, Google Assistant, or Alexa, quick responses are critical for a seamless interaction.
3. **Increased Productivity**: By automating tasks such as text classification, language translation, and text summarization, fast language models can significantly boost productivity in various industries, including customer service, content creation, and research.
4. **Real-time Insights**: 

# **Inicializando o cliente, o sistema e a mensagem do sistema**
# **Coloca a mensagem na lista de mensagens e devolve o resultado**

In [4]:
class Agent:
  def __init__(self, client, system):
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    completetion = client.chat.completions.create(
        messages= self.messages,
        model="llama-3.3-70b-versatile",
    )
    return completetion.choices[0].message.content


# **Prompt explicando a LLM o que fazer**

In [5]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 44.00 * 0.90 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_makeup_price:
e.g. get_makeup_price: Charlotte Tilbury Flawless Filter
Returns the current price of a specific makeup product.

Example session:

Question: How much would it cost to buy 3 Dior Lip Glow oils if I have a 20% off coupon?
Thought: I need to find the price of a single Dior Lip Glow oil first.
Action: get_makeup_price: Dior Lip Glow Oil
PAUSE

You will be called again with this:

Observation: 40.00

Thought: I have the price ($40.00). Now I need to calculate the total for 3 units and then apply the 20% discount (multiplying by 0.80).
Action: calculate: (40.00 * 3) * 0.80
PAUSE

You will be called again with this:

Observation: 96.0

If you have the answer, output it as the Answer.

Answer: The cost for 3 Dior Lip Glow oils with a 20% discount is $96.00.
Now it's your turn:
""".strip()

# **Serão as ferramentas que a LLM vai ter acesso**

In [6]:
# tools
def calculate(operation):
    return eval(operation)

def get_makeup_price(product: str) -> float:
    match product.lower():
        case "dior lip glow oil":
            return 40.0
        case "rare beauty blush":
            return 23.0
        case "fenty beauty foundation":
            return 40.0
        case "charlotte tilbury flawless filter":
            return 49.0
        case "nars radiant creamy concealer":
            return 32.0
        case "mac ruby woo lipstick":
            return 23.0
        case "laneige lip sleeping mask":
            return 24.0
        case "glossier boy brow":
            return 17.0
        case _:
            return 0.0

# **Define o cliente como cliente e o sistema como o prompt**

In [7]:
neil_tyson = Agent(client=client, system=system_prompt)

# **Criando o loop para o agente rodar automaticamente**

In [8]:
import re

def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt)
  tools = ["calculate", "get_makeup_price"]
  next_prompt = query
  i = 0

  while i < max_iterations:
      i += 1
      result = agent(next_prompt)
      print(result)

      if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]
            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

            if "Answer" in result:
              break

# **Dando o comando ao agente**

In [9]:

agent_loop(max_iterations=10, system=system_prompt, query="How much is 2 Dior Lip Glow Oils with 10% tax?")

Thought: To find the total cost of 2 Dior Lip Glow Oils with 10% tax, I first need to find the price of a single Dior Lip Glow Oil, then calculate the total for 2 units, and finally apply the 10% tax.

Action: get_makeup_price: Dior Lip Glow Oil
PAUSE
Observation: 40.0
Thought: I have the price ($40.00) of a single Dior Lip Glow Oil. Now, I need to calculate the total cost for 2 units and apply the 10% tax. This involves two steps: first, calculating the subtotal for 2 units, and then applying the tax to this subtotal.

Action: calculate: 40.0 * 2 * 1.10
PAUSE
Observation: 88.0
Thought: I have calculated the total cost for 2 Dior Lip Glow Oils with 10% tax, which is $88.00. This is the final step in determining the answer.

Answer: The cost for 2 Dior Lip Glow Oils with 10% tax is $88.00.
Thought: The calculation is complete, and I have the total cost for 2 Dior Lip Glow Oils with 10% tax.

Answer: The cost for 2 Dior Lip Glow Oils with 10% tax is $88.00.
Thought: The observation confi